In [28]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("API_FOOTBALL_KEY")

print(api_key is not None)  # should print True
print(len(api_key))          # should print a number, not error

True
32


In [29]:
import requests

url = "https://v3.football.api-sports.io/status"
headers = {"x-apisports-key": api_key}

response = requests.get(url, headers=headers)
data = response.json()

print(data)

{'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Andrew', 'lastname': 'Maina', 'email': 'andrewmaina384@gmail.com'}, 'subscription': {'plan': 'Free', 'end': '2027-07-17T00:00:00+00:00', 'active': True}, 'requests': {'current': 35, 'limit_day': 100}}}


In [30]:
import json
print(json.dumps(data, indent=2))

{
  "get": "status",
  "parameters": [],
  "errors": [],
  "results": 0,
  "paging": {
    "current": 1,
    "total": 1
  },
  "response": {
    "account": {
      "firstname": "Andrew",
      "lastname": "Maina",
      "email": "andrewmaina384@gmail.com"
    },
    "subscription": {
      "plan": "Free",
      "end": "2027-07-17T00:00:00+00:00",
      "active": true
    },
    "requests": {
      "current": 35,
      "limit_day": 100
    }
  }
}


In [31]:
import time

competitions_to_find = [
    "World Cup",
    "World Cup - Qualification",
    "Euro Championship",
    "Africa Cup of Nations",
    "Copa America",
    "UEFA Nations League",
    "Friendlies"
]

league_ids = {}

for name in competitions_to_find:
    url = "https://v3.football.api-sports.io/leagues"
    params = {"search": name}
    response = requests.get(url, headers=headers, params=params)
    result = response.json()["response"]
    
    for league in result:
        league_ids[league["league"]["name"]] = league["league"]["id"]
    
    time.sleep(1)  # be polite to the API, avoid hammering it

print(league_ids)

{'World Cup': 1, 'World Cup - Women': 8, 'FIFA Club World Cup': 15, 'World Cup - Qualification Intercontinental Play-offs': 37, 'World Cup - U20': 490, 'World Cup - Qualification CONCACAF': 31, 'World Cup - Qualification Europe': 32, 'World Cup - Qualification Oceania': 33, 'World Cup - Qualification South America': 34, 'World Cup - Qualification Africa': 29, 'World Cup - Qualification Asia': 30, 'World Cup - U17': 587, 'World Cup - U20 - Women': 920, 'World Cup - Women - Qualification Concacaf': 927, 'World Cup - U17 - Women': 950, 'World Cup - Women - Qualification Europe': 880, 'FIFA Club World Cup - Play-In': 1186, 'Kings World Cup Nations': 1213, 'Euro Championship': 4, 'Euro Championship - Qualification': 960, 'Africa Cup of Nations': 6, 'Africa Cup of Nations U20': 538, 'Africa Cup of Nations - Qualification': 36, 'Africa Cup of Nations - Women': 922, 'Copa America': 9, 'Copa America Femenina': 926, 'UEFA Nations League': 5, 'UEFA Nations League - Women': 1040, 'Friendlies': 10,

In [32]:
core_leagues = {
    "World Cup": 1,
    "World Cup - Qualification Europe": 32,
    "World Cup - Qualification South America": 34,
    "World Cup - Qualification Africa": 29,
    "Euro Championship": 4,
    "Africa Cup of Nations": 6,
    "Copa America": 9,
    "UEFA Nations League": 5,
    "Friendlies": 10
}

seasons = [2014, 2018, 2022]

print(f"Planned requests: {len(core_leagues) * len(seasons)}")

Planned requests: 27


In [33]:
import pandas as pd
import time
import os

all_fixtures = []

for league_name, league_id in core_leagues.items():
    for season in seasons:
        url = "https://v3.football.api-sports.io/fixtures"
        params = {"league": league_id, "season": season}
        response = requests.get(url, headers=headers, params=params)
        result = response.json()

        matches = result["response"]
        print(f"{league_name} {season}: {len(matches)} matches")

        for m in matches:
            all_fixtures.append({
                "fixture_id": m["fixture"]["id"],
                "date": m["fixture"]["date"],
                "league": league_name,
                "season": season,
                "home_team": m["teams"]["home"]["name"],
                "away_team": m["teams"]["away"]["name"],
                "home_team_id": m["teams"]["home"]["id"],
                "away_team_id": m["teams"]["away"]["id"],
                "home_goals": m["goals"]["home"],
                "away_goals": m["goals"]["away"],
                "status": m["fixture"]["status"]["short"]
            })

        time.sleep(1)

df = pd.DataFrame(all_fixtures)
print(df.shape)
df.head()

World Cup 2014: 0 matches
World Cup 2018: 0 matches
World Cup 2022: 64 matches
World Cup - Qualification Europe 2014: 0 matches
World Cup - Qualification Europe 2018: 0 matches
World Cup - Qualification Europe 2022: 0 matches
World Cup - Qualification South America 2014: 0 matches
World Cup - Qualification South America 2018: 0 matches
World Cup - Qualification South America 2022: 90 matches
World Cup - Qualification Africa 2014: 0 matches
World Cup - Qualification Africa 2018: 0 matches
World Cup - Qualification Africa 2022: 158 matches
Euro Championship 2014: 0 matches
Euro Championship 2018: 0 matches
Euro Championship 2022: 0 matches
Africa Cup of Nations 2014: 0 matches
Africa Cup of Nations 2018: 0 matches
Africa Cup of Nations 2022: 0 matches
Copa America 2014: 0 matches
Copa America 2018: 0 matches
Copa America 2022: 0 matches
UEFA Nations League 2014: 0 matches
UEFA Nations League 2018: 0 matches
UEFA Nations League 2022: 0 matches
Friendlies 2014: 0 matches
Friendlies 2018: 0

,fixture_id,date,league,season,home_team,away_team,home_team_id,away_team_id,home_goals,away_goals,status
0,855736,2022-11-20T16:00:00+00:00,World Cup,2022,Qatar,Ecuador,1569,2382,0.0,2.0,FT
1,855735,2022-11-21T13:00:00+00:00,World Cup,2022,England,Iran,10,22,6.0,2.0,FT
2,855734,2022-11-21T16:00:00+00:00,World Cup,2022,Senegal,Netherlands,13,1118,0.0,2.0,FT
3,866681,2022-11-21T19:00:00+00:00,World Cup,2022,USA,Wales,2384,767,1.0,1.0,FT
4,855737,2022-11-22T10:00:00+00:00,World Cup,2022,Argentina,Saudi Arabia,26,23,1.0,2.0,FT


In [34]:
for league_name, league_id in core_leagues.items():
    url = "https://v3.football.api-sports.io/leagues"
    params = {"id": league_id}
    response = requests.get(url, headers=headers, params=params)
    result = response.json()["response"][0]
    
    available_seasons = [s["year"] for s in result["seasons"]]
    print(f"{league_name}: {available_seasons}")
    
    time.sleep(1)

World Cup: [2010, 2014, 2018, 2022, 2026]
World Cup - Qualification Europe: [2018, 2020, 2024]
World Cup - Qualification South America: [2018, 2022, 2026]
World Cup - Qualification Africa: [2018, 2022, 2023]
Euro Championship: [2008, 2012, 2016, 2020, 2024]
Africa Cup of Nations: [2015, 2017, 2019, 2021, 2023, 2025]
Copa America: [2015, 2016, 2019, 2021, 2024]
UEFA Nations League: [2018, 2020, 2022, 2024, 2026]
Friendlies: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [35]:
selected_seasons = {
    "World Cup": [2022],
    "World Cup - Qualification Europe": [2024],
    "World Cup - Qualification South America": [2022],
    "World Cup - Qualification Africa": [2023],
    "Euro Championship": [2024],
    "Africa Cup of Nations": [2023],
    "Copa America": [2024],
    "UEFA Nations League": [2024],
    "Friendlies": [2023, 2024, 2025]
}

all_fixtures = []

for league_name, years in selected_seasons.items():
    league_id = core_leagues[league_name]
    for season in years:
        url = "https://v3.football.api-sports.io/fixtures"
        params = {"league": league_id, "season": season}
        response = requests.get(url, headers=headers, params=params)
        matches = response.json()["response"]
        print(f"{league_name} {season}: {len(matches)} matches")

        for m in matches:
            all_fixtures.append({
                "fixture_id": m["fixture"]["id"],
                "date": m["fixture"]["date"],
                "league": league_name,
                "season": season,
                "home_team": m["teams"]["home"]["name"],
                "away_team": m["teams"]["away"]["name"],
                "home_team_id": m["teams"]["home"]["id"],
                "away_team_id": m["teams"]["away"]["id"],
                "home_goals": m["goals"]["home"],
                "away_goals": m["goals"]["away"],
                "status": m["fixture"]["status"]["short"]
            })
        time.sleep(1)

df = pd.DataFrame(all_fixtures)
print(df.shape)
df.head()

World Cup 2022: 0 matches
World Cup - Qualification Europe 2024: 0 matches
World Cup - Qualification South America 2022: 0 matches
World Cup - Qualification Africa 2023: 0 matches
Euro Championship 2024: 0 matches
Africa Cup of Nations 2023: 0 matches
Copa America 2024: 0 matches
UEFA Nations League 2024: 0 matches
Friendlies 2023: 0 matches
Friendlies 2024: 0 matches
Friendlies 2025: 0 matches
(0, 0)


""


In [ ]:
# Keep only finished matches with valid scores
df_clean = df[df["status"] == "FT"].dropna(subset=["home_goals", "away_goals"]).copy()

# Add the actual prediction target
def get_result(row):
    if row["home_goals"] > row["away_goals"]:
        return "home_win"
    elif row["home_goals"] < row["away_goals"]:
        return "away_win"
    else:
        return "draw"

df_clean["result"] = df_clean.apply(get_result, axis=1)

print(df_clean.shape)
print(df_clean["result"].value_counts())

df_clean.to_csv("../data/raw/fixtures.csv", index=False)

(1654, 12)
result
home_win    782
away_win    492
draw        380
Name: count, dtype: int64


In [ ]:
df_clean = df_clean.sort_values("date").reset_index(drop=True)

home_rows = df_clean[["date", "fixture_id", "home_team_id", "home_goals", "away_goals"]].copy()
home_rows.columns = ["date", "fixture_id", "team_id", "goals_for", "goals_against"]
home_rows["points"] = home_rows.apply(
    lambda r: 3 if r["goals_for"] > r["goals_against"] else (1 if r["goals_for"] == r["goals_against"] else 0),
    axis=1
)

away_rows = df_clean[["date", "fixture_id", "away_team_id", "away_goals", "home_goals"]].copy()
away_rows.columns = ["date", "fixture_id", "team_id", "goals_for", "goals_against"]
away_rows["points"] = away_rows.apply(
    lambda r: 3 if r["goals_for"] > r["goals_against"] else (1 if r["goals_for"] == r["goals_against"] else 0),
    axis=1
)

team_matches = pd.concat([home_rows, away_rows]).sort_values(["team_id", "date"]).reset_index(drop=True)
team_matches.head(10)

,date,fixture_id,team_id,goals_for,goals_against,points
0,2022-11-23T19:00:00+00:00,855742,1,1.0,0.0,3
1,2022-11-27T13:00:00+00:00,855753,1,0.0,2.0,0
2,2022-12-01T15:00:00+00:00,855766,1,0.0,0.0,1
3,2023-03-28T18:45:00+00:00,1012344,1,3.0,2.0,3
4,2023-11-15T19:45:00+00:00,1028663,1,1.0,0.0,3
5,2024-03-23T17:00:00+00:00,1155268,1,0.0,0.0,1
6,2024-03-26T19:45:00+00:00,1155276,1,2.0,2.0,1
7,2024-06-05T18:30:00+00:00,1169677,1,2.0,0.0,3
8,2024-06-08T18:00:00+00:00,1168423,1,3.0,0.0,3
9,2024-06-17T16:00:00+00:00,1145516,1,0.0,1.0,0


In [ ]:
team_matches["form_points"] = (
    team_matches.groupby("team_id")["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

team_matches.head(10)

,date,fixture_id,team_id,goals_for,goals_against,points,form_points
0,2022-11-23T19:00:00+00:00,855742,1,1.0,0.0,3,NaN
1,2022-11-27T13:00:00+00:00,855753,1,0.0,2.0,0,3.0
2,2022-12-01T15:00:00+00:00,855766,1,0.0,0.0,1,3.0
3,2023-03-28T18:45:00+00:00,1012344,1,3.0,2.0,3,4.0
4,2023-11-15T19:45:00+00:00,1028663,1,1.0,0.0,3,7.0
5,2024-03-23T17:00:00+00:00,1155268,1,0.0,0.0,1,10.0
6,2024-03-26T19:45:00+00:00,1155276,1,2.0,2.0,1,8.0
7,2024-06-05T18:30:00+00:00,1169677,1,2.0,0.0,3,9.0
8,2024-06-08T18:00:00+00:00,1168423,1,3.0,0.0,3,11.0
9,2024-06-17T16:00:00+00:00,1145516,1,0.0,1.0,0,11.0


In [ ]:
# Merge form for the home team
df_clean = df_clean.merge(
    team_matches[["fixture_id", "team_id", "form_points"]],
    left_on=["fixture_id", "home_team_id"],
    right_on=["fixture_id", "team_id"],
    how="left"
).rename(columns={"form_points": "home_form"}).drop(columns=["team_id"])

# Merge form for the away team
df_clean = df_clean.merge(
    team_matches[["fixture_id", "team_id", "form_points"]],
    left_on=["fixture_id", "away_team_id"],
    right_on=["fixture_id", "team_id"],
    how="left"
).rename(columns={"form_points": "away_form"}).drop(columns=["team_id"])

df_clean[["date", "home_team", "away_team", "home_form", "away_form", "result"]].head(10)

,date,home_team,away_team,home_form,away_form,result
0,2020-10-08T22:30:00+00:00,Paraguay,Peru,NaN,NaN,draw
1,2020-10-08T22:45:00+00:00,Uruguay,Chile,NaN,NaN,home_win
2,2020-10-09T00:30:00+00:00,Argentina,Ecuador,NaN,NaN,home_win
3,2020-10-09T23:30:00+00:00,Colombia,Venezuela,NaN,NaN,home_win
4,2020-10-10T00:30:00+00:00,Brazil,Bolivia,NaN,NaN,home_win
5,2020-10-13T20:00:00+00:00,Bolivia,Argentina,0.0,3.0,away_win
6,2020-10-13T21:00:00+00:00,Ecuador,Uruguay,0.0,3.0,home_win
7,2020-10-14T00:00:00+00:00,Peru,Brazil,1.0,3.0,away_win
8,2020-10-14T00:00:00+00:00,Venezuela,Paraguay,0.0,1.0,away_win
9,2020-10-14T00:30:00+00:00,Chile,Colombia,0.0,3.0,draw


In [ ]:
print(df_clean[["home_form", "away_form"]].isnull().sum())
print(df_clean.shape)

home_form    175
away_form    178
dtype: int64
(1654, 14)


In [ ]:
df_model = df_clean.dropna(subset=["home_form", "away_form"]).copy()
print(df_model.shape)
print(df_model["result"].value_counts())

(1419, 14)
result
home_win    662
away_win    439
draw        318
Name: count, dtype: int64


In [ ]:
import numpy as np

df_model = df_model.sort_values("date").reset_index(drop=True)

def get_h2h(row, all_matches):
    past = all_matches[
        (all_matches["date"] < row["date"]) &
        (
            ((all_matches["home_team_id"] == row["home_team_id"]) & (all_matches["away_team_id"] == row["away_team_id"])) |
            ((all_matches["home_team_id"] == row["away_team_id"]) & (all_matches["away_team_id"] == row["home_team_id"]))
        )
    ]

    if len(past) == 0:
        return pd.Series([np.nan, 0])

    home_team_wins = 0
    for _, m in past.iterrows():
        if m["home_team_id"] == row["home_team_id"] and m["result"] == "home_win":
            home_team_wins += 1
        elif m["away_team_id"] == row["home_team_id"] and m["result"] == "away_win":
            home_team_wins += 1

    return pd.Series([home_team_wins / len(past), len(past)])

df_model[["h2h_home_win_rate", "h2h_matches_played"]] = df_model.apply(
    lambda row: get_h2h(row, df_model), axis=1
)

df_model[["date", "home_team", "away_team", "h2h_home_win_rate", "h2h_matches_played"]].head(15)

,date,home_team,away_team,h2h_home_win_rate,h2h_matches_played
0,2020-10-13T20:00:00+00:00,Bolivia,Argentina,NaN,0.0
1,2020-10-13T21:00:00+00:00,Ecuador,Uruguay,NaN,0.0
2,2020-10-14T00:00:00+00:00,Peru,Brazil,NaN,0.0
3,2020-10-14T00:00:00+00:00,Venezuela,Paraguay,NaN,0.0
4,2020-10-14T00:30:00+00:00,Chile,Colombia,NaN,0.0
5,2020-11-12T20:00:00+00:00,Bolivia,Ecuador,NaN,0.0
6,2020-11-13T00:00:00+00:00,Argentina,Paraguay,NaN,0.0
7,2020-11-13T20:30:00+00:00,Colombia,Uruguay,NaN,0.0
8,2020-11-13T23:00:00+00:00,Chile,Peru,NaN,0.0
9,2020-11-14T00:30:00+00:00,Brazil,Venezuela,NaN,0.0


In [ ]:
print(df_model["h2h_matches_played"].value_counts().sort_index())
print()
print(f"Matches with zero prior h2h: {(df_model['h2h_matches_played'] == 0).sum()} out of {len(df_model)}")

h2h_matches_played
0.0    981
1.0    384
2.0     46
3.0      7
4.0      1
Name: count, dtype: int64

Matches with zero prior h2h: 981 out of 1419


In [ ]:
df_model["h2h_home_win_rate"] = df_model["h2h_home_win_rate"].fillna(0.5)

print(df_model[["h2h_home_win_rate", "h2h_matches_played"]].isnull().sum())
print(df_model.shape)

h2h_home_win_rate     0
h2h_matches_played    0
dtype: int64
(1419, 16)


In [ ]:
df_model = df_model.sort_values("date").reset_index(drop=True)

split_index = int(len(df_model) * 0.8)
train = df_model.iloc[:split_index]
test = df_model.iloc[split_index:]

print(f"Train: {len(train)} matches, up to {train['date'].max()}")
print(f"Test: {len(test)} matches, from {test['date'].min()}")

Train: 1135 matches, up to 2025-06-06T18:45:00+00:00
Test: 284 matches, from 2025-06-06T18:45:00+00:00


## TRAINING THE DATA
* Train: 1135 matches
* Test: 284 matches

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

features = ["home_form", "away_form", "h2h_home_win_rate", "h2h_matches_played"]

le = LabelEncoder()
df_model["result_encoded"] = le.fit_transform(df_model["result"])

X_train = train[features]
y_train = df_model.loc[train.index, "result_encoded"]
X_test = test[features]
y_test = df_model.loc[test.index, "result_encoded"]

model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")
print()
print(classification_report(y_test, predictions, target_names=le.classes_))

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/andrewmaina/Documents/Github projects/World_cup_predictor/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: '@rpath/libomp.dylib'\n  Referenced from: '/Users/andrewmaina/Documents/Github projects/World_cup_predictor/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib'\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/lib/libomp.dylib' (no such file)"]


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

features = ["home_form", "away_form", "h2h_home_win_rate", "h2h_matches_played"]

le = LabelEncoder()
df_model["result_encoded"] = le.fit_transform(df_model["result"])

X_train = train[features]
y_train = df_model.loc[train.index, "result_encoded"]
X_test = test[features]
y_test = df_model.loc[test.index, "result_encoded"]

model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")
print()
print(classification_report(y_test, predictions, target_names=le.classes_))

Accuracy: 51.41%

              precision    recall  f1-score   support

    away_win       0.59      0.42      0.49       106
        draw       0.15      0.19      0.17        47
    home_win       0.62      0.70      0.66       131

    accuracy                           0.51       284
   macro avg       0.45      0.44      0.44       284
weighted avg       0.53      0.51      0.52       284



In [ ]:
baseline_accuracy = (test["result"] == "home_win").mean()
print(f"Always guessing home win: {baseline_accuracy:.2%}")

Always guessing home win: 46.13%


In [ ]:
def calculate_elo(matches, k=30, base_rating=1500):
    ratings = {}
    elo_home, elo_away = [], []

    for _, row in matches.iterrows():
        home_id, away_id = row["home_team_id"], row["away_team_id"]
        home_elo = ratings.get(home_id, base_rating)
        away_elo = ratings.get(away_id, base_rating)

        elo_home.append(home_elo)
        elo_away.append(away_elo)

        expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))

        if row["result"] == "home_win":
            actual_home = 1
        elif row["result"] == "away_win":
            actual_home = 0
        else:
            actual_home = 0.5

        ratings[home_id] = home_elo + k * (actual_home - expected_home)
        ratings[away_id] = away_elo + k * ((1 - actual_home) - (1 - expected_home))

    matches["home_elo"] = elo_home
    matches["away_elo"] = elo_away
    return matches

df_model = calculate_elo(df_model)
df_model[["date", "home_team", "away_team", "home_elo", "away_elo", "result"]].tail(10)

,date,home_team,away_team,home_elo,away_elo,result
1409,2026-03-26T17:00:00+00:00,Gibraltar,Latvia,1392.756459,1401.964818,away_win
1410,2026-03-26T19:45:00+00:00,Slovakia,Kosovo,1573.419388,1558.596959,away_win
1411,2026-03-26T19:45:00+00:00,Ukraine,Sweden,1520.104929,1513.744751,away_win
1412,2026-03-26T19:45:00+00:00,Poland,Albania,1527.077090,1504.418699,home_win
1413,2026-03-26T19:45:00+00:00,Denmark,FYR Macedonia,1545.426385,1543.239035,home_win
1414,2026-03-26T19:45:00+00:00,Italy,Northern Ireland,1582.511288,1532.439228,home_win
1415,2026-03-31T16:00:00+00:00,Luxembourg,Malta,1413.165603,1426.378008,home_win
1416,2026-03-31T16:00:00+00:00,Latvia,Gibraltar,1416.567354,1378.153923,home_win
1417,2026-03-31T18:45:00+00:00,Sweden,Poland,1529.019311,1541.100233,home_win
1418,2026-03-31T18:45:00+00:00,Kosovo,Türkiye,1574.236507,1602.967396,away_win


In [ ]:
features_v2 = ["home_form", "away_form", "h2h_home_win_rate", "h2h_matches_played", "home_elo", "away_elo"]

X_train_v2 = train_v2 = df_model.loc[train.index, features_v2]
X_test_v2 = df_model.loc[test.index, features_v2]

model_v2 = HistGradientBoostingClassifier(random_state=42)
model_v2.fit(X_train_v2, y_train)

train_preds_v2 = model_v2.predict(X_train_v2)
test_preds_v2 = model_v2.predict(X_test_v2)

print(f"Train accuracy: {accuracy_score(y_train, train_preds_v2):.2%}")
print(f"Test accuracy:  {accuracy_score(y_test, test_preds_v2):.2%}")
print()
print(classification_report(y_test, test_preds_v2, target_names=le.classes_))

Train accuracy: 92.69%
Test accuracy:  53.17%

              precision    recall  f1-score   support

    away_win       0.51      0.53      0.52       106
        draw       0.20      0.15      0.17        47
    home_win       0.63      0.67      0.65       131

    accuracy                           0.53       284
   macro avg       0.45      0.45      0.45       284
weighted avg       0.52      0.53      0.52       284



In [37]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model_v2, X_test_v2, y_test, n_repeats=10, random_state=42
)

importances = pd.Series(result.importances_mean, index=features_v2).sort_values(ascending=False)
print(importances)

home_elo              0.103169
away_elo              0.045070
h2h_home_win_rate    -0.001761
away_form            -0.007394
h2h_matches_played   -0.023239
home_form            -0.026056
dtype: float64


In [38]:
model_v1_reg = HistGradientBoostingClassifier(
    random_state=42, max_depth=3, max_iter=100, l2_regularization=1.0
)
model_v1_reg.fit(X_train, y_train)

model_v2_reg = HistGradientBoostingClassifier(
    random_state=42, max_depth=3, max_iter=100, l2_regularization=1.0
)
model_v2_reg.fit(X_train_v2, y_train)

# Elo-only, dropping the noisy features entirely
features_elo_only = ["home_elo", "away_elo"]
X_train_elo = df_model.loc[train.index, features_elo_only]
X_test_elo = df_model.loc[test.index, features_elo_only]

model_v3_reg = HistGradientBoostingClassifier(
    random_state=42, max_depth=3, max_iter=100, l2_regularization=1.0
)
model_v3_reg.fit(X_train_elo, y_train)

for name, model, Xtr, Xte in [
    ("v1: form + h2h only", model_v1_reg, X_train, X_test),
    ("v2: form + h2h + elo", model_v2_reg, X_train_v2, X_test_v2),
    ("v3: elo only", model_v3_reg, X_train_elo, X_test_elo)
]:
    train_acc = accuracy_score(y_train, model.predict(Xtr))
    test_acc = accuracy_score(y_test, model.predict(Xte))
    print(f"{name}: train {train_acc:.2%}, test {test_acc:.2%}")

v1: form + h2h only: train 53.74%, test 59.15%
v2: form + h2h + elo: train 62.73%, test 60.21%
v3: elo only: train 58.24%, test 57.39%


In [39]:
test_preds_v2_reg = model_v2_reg.predict(X_test_v2)
print(classification_report(y_test, test_preds_v2_reg, target_names=le.classes_))

              precision    recall  f1-score   support

    away_win       0.60      0.63      0.62       106
        draw       0.30      0.15      0.20        47
    home_win       0.65      0.74      0.69       131

    accuracy                           0.60       284
   macro avg       0.52      0.51      0.50       284
weighted avg       0.57      0.60      0.58       284



In [40]:
team_matches["goal_diff"] = team_matches["goals_for"] - team_matches["goals_against"]

team_matches["form_goal_diff"] = (
    team_matches.groupby("team_id")["goal_diff"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto df_model, same pattern as before
df_model = df_model.merge(
    team_matches[["fixture_id", "team_id", "form_goal_diff"]],
    left_on=["fixture_id", "home_team_id"], right_on=["fixture_id", "team_id"], how="left"
).rename(columns={"form_goal_diff": "home_goal_diff"}).drop(columns=["team_id"])

df_model = df_model.merge(
    team_matches[["fixture_id", "team_id", "form_goal_diff"]],
    left_on=["fixture_id", "away_team_id"], right_on=["fixture_id", "team_id"], how="left"
).rename(columns={"form_goal_diff": "away_goal_diff"}).drop(columns=["team_id"])

df_model["home_goal_diff"] = df_model["home_goal_diff"].fillna(0)
df_model["away_goal_diff"] = df_model["away_goal_diff"].fillna(0)

df_model[["home_team", "away_team", "home_goal_diff", "away_goal_diff", "result"]].tail(10)

,home_team,away_team,home_goal_diff,away_goal_diff,result
1409,Gibraltar,Latvia,-2.4,-1.6,away_win
1410,Slovakia,Kosovo,-0.8,1.0,away_win
1411,Ukraine,Sweden,0.2,-1.6,away_win
1412,Poland,Albania,1.0,0.2,home_win
1413,Denmark,FYR Macedonia,1.8,0.0,home_win
1414,Italy,Northern Ireland,1.0,-0.2,home_win
1415,Luxembourg,Malta,-1.4,-1.2,home_win
1416,Latvia,Gibraltar,-1.2,-2.4,home_win
1417,Sweden,Poland,-0.8,1.2,home_win
1418,Kosovo,Türkiye,0.8,2.2,away_win


In [41]:
features_v4 = ["home_form", "away_form", "h2h_home_win_rate", "h2h_matches_played",
               "home_elo", "away_elo", "home_goal_diff", "away_goal_diff"]

X_train_v4 = df_model.loc[train.index, features_v4]
X_test_v4 = df_model.loc[test.index, features_v4]

model_v4_reg = HistGradientBoostingClassifier(
    random_state=42, max_depth=3, max_iter=100, l2_regularization=1.0
)
model_v4_reg.fit(X_train_v4, y_train)

train_acc = accuracy_score(y_train, model_v4_reg.predict(X_train_v4))
test_acc = accuracy_score(y_test, model_v4_reg.predict(X_test_v4))
print(f"Train accuracy: {train_acc:.2%}")
print(f"Test accuracy:  {test_acc:.2%}")
print()
print(classification_report(y_test, model_v4_reg.predict(X_test_v4), target_names=le.classes_))

Train accuracy: 64.76%
Test accuracy:  61.27%

              precision    recall  f1-score   support

    away_win       0.59      0.66      0.62       106
        draw       0.31      0.11      0.16        47
    home_win       0.66      0.76      0.71       131

    accuracy                           0.61       284
   macro avg       0.52      0.51      0.50       284
weighted avg       0.58      0.61      0.58       284



In [42]:
import joblib

joblib.dump(model_v2_reg, "../src/model.pkl")
joblib.dump(le, "../src/label_encoder.pkl")
print("Model saved")

Model saved


In [45]:
def get_latest_team_stats(team_id, team_matches_df):
    team_history = team_matches_df[team_matches_df["team_id"] == team_id].sort_values("date")
    
    if len(team_history) == 0:
        return {"form": 0, "elo": 1500}
    
    latest = team_history.iloc[-1]
    recent_5 = team_history.tail(5)
    form = recent_5["points"].sum()
    
    return {"form": form, "elo": latest.get("elo", 1500)}

In [46]:
# Build a clean "latest known state" table, one row per team, easy to query at prediction time
latest_home = df_model.sort_values("date").groupby("home_team_id").last()[["home_elo"]]
latest_away = df_model.sort_values("date").groupby("away_team_id").last()[["away_elo"]]
latest_home.columns = ["elo"]
latest_away.columns = ["elo"]

latest_elo = pd.concat([latest_home, latest_away]).groupby(level=0).last()

latest_form = team_matches.sort_values("date").groupby("team_id").last()[["form_points"]]

team_id_to_name = pd.concat([
    df_model[["home_team_id", "home_team"]].rename(columns={"home_team_id": "id", "home_team": "name"}),
    df_model[["away_team_id", "away_team"]].rename(columns={"away_team_id": "id", "away_team": "name"})
]).drop_duplicates(subset="id").set_index("id")["name"]

team_id_to_name.to_csv("../data/raw/team_names.csv")
latest_elo.to_csv("../data/raw/latest_elo.csv")
latest_form.to_csv("../data/raw/latest_form.csv")

In [47]:
print(latest_elo.head())
print()
print(team_id_to_name.head())
print()
print(latest_form.head())

import os
print(os.path.exists("../data/raw/team_names.csv"))
print(os.path.exists("../data/raw/latest_elo.csv"))
print(os.path.exists("../data/raw/latest_form.csv"))

           elo
1  1546.487248
2  1614.551267
3  1613.983283
4  1555.336528
5  1513.744751

id
2381      Bolivia
2382      Ecuador
30           Peru
2379    Venezuela
2383        Chile
Name: name, dtype: object

         form_points
team_id             
1               11.0
2               13.0
3               13.0
4               15.0
5                4.0
True
True
True


In [48]:
def get_h2h_key(id1, id2):
    return tuple(sorted([id1, id2]))

df_model["h2h_key"] = df_model.apply(lambda r: get_h2h_key(r["home_team_id"], r["away_team_id"]), axis=1)

latest_h2h = (
    df_model.sort_values("date")
    .groupby("h2h_key")
    .last()[["h2h_home_win_rate", "h2h_matches_played"]]
)

latest_h2h.to_csv("../data/raw/latest_h2h.csv")
print(latest_h2h.head())
print(os.path.exists("../data/raw/latest_h2h.csv"))

         h2h_home_win_rate  h2h_matches_played
h2h_key                                       
(1, 2)                 0.0                 2.0
(1, 3)                 0.5                 0.0
(1, 10)                0.5                 0.0
(1, 14)                0.5                 0.0
(1, 25)                0.5                 0.0
True


In [49]:
# Rebuild h2h lookup with plain columns, no tuple keys
df_model["team_a"] = df_model.apply(lambda r: min(r["home_team_id"], r["away_team_id"]), axis=1)
df_model["team_b"] = df_model.apply(lambda r: max(r["home_team_id"], r["away_team_id"]), axis=1)

latest_h2h = (
    df_model.sort_values("date")
    .groupby(["team_a", "team_b"])
    .last()[["h2h_home_win_rate", "h2h_matches_played", "home_team_id"]]
    .reset_index()
)
latest_h2h.to_csv("../data/raw/latest_h2h.csv", index=False)

# Re-save elo and form with explicit, named index columns
latest_elo.index.name = "team_id"
latest_elo.to_csv("../data/raw/latest_elo.csv")

latest_form.to_csv("../data/raw/latest_form.csv")  # already named team_id, confirmed by your output

print(latest_h2h.head())

   team_a  team_b  h2h_home_win_rate  h2h_matches_played  home_team_id
0       1       2                0.0                 2.0             1
1       1       3                0.5                 0.0             3
2       1      10                0.5                 0.0            10
3       1      14                0.5                 0.0             1
4       1      25                0.5                 0.0            25
